El objetivo de este cuaderno es hacer un ensemble entre el mejor modelo de los cuadernos *LLM.ipynb* y *Transformers.ipynb*, junto con un modelo que trabaje con imágenes y otro modelo que trabaje con videos.

En este cuaderno también haremos un análisis de errores.

Dentro de los diferentes tipos de ensembles que podemos hacer nos decantamos por la opción de *Predicción estática* ya que es la opción que se suele hacer en las competiciones dentro del mundo de NLP y en la ciencia de datos porque nos permite ajustes los pesos miles de veces en un segundo sin tener que volver a tirar de tarjeta gráfica para procesar los videos de nuevo.

Los modelos que componen el ensemble son:
- *Mistral* que ha sido el mejor modelo de entre los Transformers y LLM ala hora de evaluar el texto.
- *Convext* entrenando con el dataset de 4 frames como modelo de imágenes.
- *Timesformer* como modelo de video.

Más adelante hemos preparado una celda que calcula cual es la mejor combinación de los pesos de cada modelo al ensemble para obtener los mejores resultados contra el fichero de test estático.

In [14]:
!pip install decord
!pip install -U torchao

In [15]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import os
from tqdm import tqdm
from datasets import Dataset, Image

In [16]:
from peft import PeftModel
from PIL import Image as PILImage
from decord import VideoReader, cpu
import decord

In [17]:
# Dependencias específicas de modelos
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    ViTImageProcessor,
    ViTForImageClassification,
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification
)

In [18]:
decord.bridge.set_bridge('torch')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Definición de las rutas de los archivos

Definiendo cuáles son las rutas de los archivos .csv sobre los que vamos a hacer el ensemble

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Rutas de datos
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
CSV_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Frames/Frames-4/dataset_imagenes_test_4_Frames.csv"
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/"

In [21]:
# Rutas donde tengo guardados los modelos
DIR_MODELO_TEXTO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA/checkpoint-565"
DIR_MODELO_IMAGEN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/CONVNEXT_Frames_4/modelo_final"
DIR_MODELO_VIDEO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/TIMESFORMER_FineTuned/modelo_final"

In [22]:
# Rutas para guardar las predicciones temporales y resultados
DIR_RESULTADOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Ensemble"
os.makedirs(DIR_RESULTADOS, exist_ok=True)
RUTA_PRED_TEXTO = os.path.join(DIR_RESULTADOS, "predicciones_test_mistral.csv")
RUTA_PRED_IMAGEN = os.path.join(DIR_RESULTADOS, "predicciones_test_convnext.csv")
RUTA_PRED_VIDEO = os.path.join(DIR_RESULTADOS, "predicciones_test_timesformer.csv")

# 2. Carga del dataset de test (común para todos)

In [23]:
print("Cargando el dataset de test fijo...")
test_df = pd.read_csv(CSV_TEST_TEXT)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

#test_df

Cargando el dataset de test fijo...


# 3. Fase de generación de predicciones

# 3.1. Predicciones de texto (*Mistral*)

In [24]:
if not os.path.exists(RUTA_PRED_TEXTO):
    print("\n--- Generando predicciones de Texto (Mistral) ---")

    # 1. Cargar Tokenizador y Modelo Base
    model_id = "mistralai/Mistral-7B-Instruct-v0.3"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=2, device_map="auto", torch_dtype=torch.bfloat16
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id

    # 2. Cargar Pesos QLoRA
    model = PeftModel.from_pretrained(base_model, DIR_MODELO_TEXTO)
    model.eval()

    predicciones_texto = []

    # 3. Inferencia
    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferencia Mistral"):
        inputs = tokenizer(row["text"], return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            # Aplicamos Softmax para obtener la probabilidad de clase 1
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

        predicciones_texto.append({
            "id_EXIST": row["id_EXIST"],
            "prob_texto": prob_misogino
        })

    pd.DataFrame(predicciones_texto).to_csv(RUTA_PRED_TEXTO, index=False)

    # Limpiamos memoria
    del model, base_model, tokenizer, inputs, outputs
    torch.cuda.empty_cache()
    print("Predicciones de texto guardadas.")

## 3.2. Predicciones imagen (*ConvNext + Mean-Pooling*)

In [25]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

if not os.path.exists(RUTA_PRED_IMAGEN):
    print("\n--- Generando predicciones de Imagen (ConvNeXt - Mean-Pooling) ---")

    # Usamos Auto* para que cargue la arquitectura ConvNeXt automáticamente
    processor_img = AutoImageProcessor.from_pretrained(DIR_MODELO_IMAGEN)
    model_img = AutoModelForImageClassification.from_pretrained(DIR_MODELO_IMAGEN).to(device)
    model_img.eval()

    df_imagenes = pd.read_csv(CSV_IMAGENES)
    test_ids = test_df['id_EXIST'].unique()
    test_img_df = df_imagenes[df_imagenes['id_EXIST'].isin(test_ids)].copy()

    predicciones_por_video = {}

    for index, row in tqdm(test_img_df.iterrows(), total=len(test_img_df), desc="Inferencia ConvNeXt (Frames)"):
        id_vid = row['id_EXIST']
        try:
            image = PILImage.open(row['path_imagen']).convert("RGB")
            inputs = processor_img(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model_img(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()

            if id_vid not in predicciones_por_video:
                predicciones_por_video[id_vid] = []
            predicciones_por_video[id_vid].append(prob_misogino)
        except Exception as e:
            continue

    # Aplicar Mean-Pooling
    predicciones_img_final = []
    for id_vid, probabilidades in predicciones_por_video.items():
        total_frames = len(probabilidades)
        # Hacemos la media de las probabilidades de los fotogramas del vídeo
        prob_mean = sum(probabilidades) / total_frames if total_frames > 0 else 0.5

        predicciones_img_final.append({
            "id_EXIST": id_vid,
            "prob_imagen": prob_mean
        })

    pd.DataFrame(predicciones_img_final).to_csv(RUTA_PRED_IMAGEN, index=False)

    del model_img, processor_img
    try: del inputs, outputs
    except: pass
    torch.cuda.empty_cache()
    print("✅ Predicciones de imagen (ConvNeXt Mean-Pooling) guardadas.")


--- Generando predicciones de Imagen (ConvNeXt - Mean-Pooling) ---


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

Inferencia ConvNeXt (Frames): 100%|██████████| 2008/2008 [13:14<00:00,  2.53it/s]

✅ Predicciones de imagen (ConvNeXt Mean-Pooling) guardadas.


## 3.3. Predicciones de vídeo (*VideoMAE*)

In [26]:
from transformers import AutoImageProcessor, AutoModelForVideoClassification

if not os.path.exists(RUTA_PRED_VIDEO):
    print("\n--- Generando predicciones de Vídeo (TimeSformer Baseline) ---")

    # Usamos Auto* para que cargue la arquitectura TimeSformer sin problemas
    processor_vid = AutoImageProcessor.from_pretrained(DIR_MODELO_VIDEO)
    model_vid = AutoModelForVideoClassification.from_pretrained(DIR_MODELO_VIDEO).to(device)
    model_vid.eval()

    def sample_frame_indices(clip_len, total_frames):
        if total_frames <= clip_len:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
        else:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

    predicciones_video = []

    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferencia TimeSformer"):
        id_vid = row["id_EXIST"]
        # Construir ruta del mp4
        val = str(row.get('path_video', id_vid))
        if not val.endswith(".mp4"): val += ".mp4"
        ruta_video = os.path.join(RUTA_BASE_VIDEOS, val)

        prob_misogino = 0.5 # Valor de incertidumbre por defecto

        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(16, len(vr)) # TimeSformer usa 16 frames
            frames = vr.get_batch(frame_indices).numpy()

            inputs = processor_vid(list(frames), return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device)

            with torch.no_grad():
                outputs = model_vid(pixel_values=pixel_values)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()

        except Exception as e:
            # Si el video falla (corrupto/filtro), se queda en 0.5. Silenciamos el print para no romper la barra.
            pass

        predicciones_video.append({
            "id_EXIST": id_vid,
            "prob_video": prob_misogino # ¡Importante mantener este nombre de columna para el ensemble!
        })

    pd.DataFrame(predicciones_video).to_csv(RUTA_PRED_VIDEO, index=False)

    del model_vid, processor_vid
    try: del inputs, outputs
    except: pass
    torch.cuda.empty_cache()
    print("✅ Predicciones de vídeo (TimeSformer) guardadas.")


--- Generando predicciones de Vídeo (TimeSformer Baseline) ---


Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

Inferencia TimeSformer: 100%|██████████| 502/502 [00:00<00:00, 2430.50it/s]


✅ Predicciones de vídeo (TimeSformer) guardadas.


# 4. Ensemble multimodal

In [27]:
print("\n--- Ejecutando Ensemble ---")
df_mistral = pd.read_csv(RUTA_PRED_TEXTO)
df_convnext = pd.read_csv(RUTA_PRED_IMAGEN)
df_timesformer = pd.read_csv(RUTA_PRED_VIDEO)

# Unimos todo usando el 'id_EXIST'
df_ensemble = test_df[['id_EXIST', 'label', 'text']].merge(df_mistral, on="id_EXIST", how="left")
df_ensemble = df_ensemble.merge(df_convnext, on="id_EXIST", how="left")
df_ensemble = df_ensemble.merge(df_timesformer, on="id_EXIST", how="left")


--- Ejecutando Ensemble ---


In [28]:
# Rellenar posibles NaNs con 0.5 (incertidumbre) en caso de que algún modelo fallara en un id
df_ensemble = df_ensemble.fillna(0.5)

In [29]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

print("\n--- Buscando la mejor combinación de pesos ---")

y_true = df_ensemble['label']
mejor_f1 = 0
mejores_pesos = (0, 0, 0)
mejor_y_pred = []
mejor_prob_final = []

# Probamos combinaciones en saltos de 0.05 (5%)
for w_texto in np.arange(0, 1.05, 0.05):
    for w_imagen in np.arange(0, 1.05 - w_texto, 0.05):
        w_video = round(1.0 - w_texto - w_imagen, 2)

        # Filtro de seguridad por los redondeos de coma flotante
        if w_video < 0 or w_video > 1:
            continue

        # Calculamos la probabilidad final con los pesos actuales
        prob_final = (
            (df_ensemble['prob_texto'] * w_texto) +
            (df_ensemble['prob_imagen'] * w_imagen) +
            (df_ensemble['prob_video'] * w_video)
        )

        # Predicción binaria
        y_pred_actual = (prob_final > 0.5).astype(int)

        # Evaluamos
        f1_actual = f1_score(y_true, y_pred_actual, average='macro')

        # Si mejoramos, guardamos el récord y las predicciones
        if f1_actual > mejor_f1:
            mejor_f1 = f1_actual
            mejores_pesos = (w_texto, w_imagen, w_video)
            mejor_y_pred = y_pred_actual
            mejor_prob_final = prob_final


--- Buscando la mejor combinación de pesos ---


In [30]:
# ==========================================
# APLICAMOS LOS MEJORES RESULTADOS AL DATAFRAME
# ==========================================
df_ensemble['prob_final'] = mejor_prob_final
df_ensemble['prediccion_ensemble'] = mejor_y_pred
y_pred = mejor_y_pred

w_texto, w_imagen, w_video = mejores_pesos

print(f"\n🏆 ¡Búsqueda completada!")
print(f"Distribución ideal -> Texto: {w_texto:.2f} | Imagen: {w_imagen:.2f} | Vídeo: {w_video:.2f}")


🏆 ¡Búsqueda completada!
Distribución ideal -> Texto: 0.35 | Imagen: 0.55 | Vídeo: 0.10


In [31]:
# Predicción binaria final
#df_ensemble['prediccion_ensemble'] = (df_ensemble['prob_final'] > 0.5).astype(int)

# 5. Evaluación del ensemble

In [32]:
=============
print("\n" + "="*50)
print("🏆 RESULTADOS DEL MODELO ENSEMBLE MULTIMODAL (PESOS ÓPTIMOS)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))

SyntaxError: invalid syntax (1305959855.py, line 1)

In [ ]:
# Guardar las predicciones completas del Ensemble Multimodal
ruta_ensemble_final = os.path.join(DIR_RESULTADOS, "predicciones_ensemble_final.csv")
df_ensemble.to_csv(ruta_ensemble_final, index=False)

print(f"✅ ¡Resultados completos del Ensemble guardados en: {ruta_ensemble_final}!")

# 6. Análisis de errores

In [ ]:
print("\n🔍 Generando archivo para el Análisis de Errores...")

# Filtrar donde el Ensemble se equivocó
df_errores = df_ensemble[df_ensemble['label'] != df_ensemble['prediccion_ensemble']].copy()

def clasificar_error(row):
    if row['label'] == 1 and row['prediccion_ensemble'] == 0:
        return "Falso Negativo (No lo detectó)"
    elif row['label'] == 0 and row['prediccion_ensemble'] == 1:
        return "Falso Positivo (Alarma falsa)"

df_errores['tipo_error'] = df_errores.apply(clasificar_error, axis=1)

In [ ]:
# Ordenamos por la 'confianza' del modelo para ver los errores más graves primero
# Error grave = probabilidad muy lejana a 0.5 (ej. Falso Positivo con 0.99 de probabilidad)
df_errores['severidad_error'] = abs(df_errores['prob_final'] - 0.5)
df_errores = df_errores.sort_values(by=['tipo_error', 'severidad_error'], ascending=[True, False])

# Reordenar columnas para que sea fácil de leer en Excel
columnas_excel = [
    'id_EXIST', 'tipo_error', 'label', 'prediccion_ensemble', 'prob_final',
    'prob_texto', 'prob_imagen', 'prob_video', 'text'
]
df_errores = df_errores[columnas_excel]

In [ ]:
ruta_errores = os.path.join(DIR_RESULTADOS, "casos_para_analisis_errores.xlsx")
df_errores.to_excel(ruta_errores, index=False)

print(f"✅ Se han encontrado {len(df_errores)} errores.")
print(f"📁 Archivo Excel de errores guardado en: {ruta_errores}")
print("¡Abre el Excel para analizar cualitativamente dónde falla tu arquitectura multimodal!")